## AUTOML **AutoGluon Tabular**

In [1]:
!pip -q install -U autogluon

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 18.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.9/98.9 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 452.1/452.1 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.8/244.8 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 8.1 MB/s eta 0:0

### Carga de datos

In [2]:
import os
import pandas as pd

CANDIDATES = [
    "../bank-full.csv",
    "./bank-full.csv",
    "/content/bank-full.csv",
]

csv_path = next((p for p in CANDIDATES if os.path.exists(p)), None)

if csv_path is None:
    try:
        from google.colab import files
        print("No encuentro bank-full.csv. Sube el fichero ahora (bank-full.csv)...")
        uploaded = files.upload()
        if "bank-full.csv" in uploaded:
            csv_path = "bank-full.csv"
    except Exception:
        pass

if csv_path is None:
    raise FileNotFoundError(
        "No se encontró bank-full.csv. Colócalo en la ruta ../bank-full.csv o ./bank-full.csv "
        "o súbelo en Colab con files.upload()."
    )

df_bank = pd.read_csv(csv_path, sep=";")
df_bank.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [3]:
df_bank.y.value_counts()

,count
y,
no,39922
yes,5289


In [4]:
df_bank.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
age,45211.0,NaN,NaN,NaN,40.93621,10.618762,18.0,33.0,39.0,48.0,95.0
job,45211,12,blue-collar,9732,NaN,NaN,NaN,NaN,NaN,NaN,NaN
marital,45211,3,married,27214,NaN,NaN,NaN,NaN,NaN,NaN,NaN
education,45211,4,secondary,23202,NaN,NaN,NaN,NaN,NaN,NaN,NaN
default,45211,2,no,44396,NaN,NaN,NaN,NaN,NaN,NaN,NaN
balance,45211.0,NaN,NaN,NaN,1362.272058,3044.765829,-8019.0,72.0,448.0,1428.0,102127.0
housing,45211,2,yes,25130,NaN,NaN,NaN,NaN,NaN,NaN,NaN
loan,45211,2,no,37967,NaN,NaN,NaN,NaN,NaN,NaN,NaN
contact,45211,3,cellular,29285,NaN,NaN,NaN,NaN,NaN,NaN,NaN
day,45211.0,NaN,NaN,NaN,15.806419,8.322476,1.0,8.0,16.0,21.0,31.0


In [5]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df_bank,
    test_size=0.2,
    random_state=123,
    stratify=df_bank["y"]
)

train_df.shape, test_df.shape

((36168, 17), (9043, 17))

### Entrenar AutoGluon

In [6]:
from autogluon.tabular import TabularPredictor

label = "y"

predictor = TabularPredictor(
    label=label,
    eval_metric="roc_auc",
    path="AutogluonModels-bank"
).fit(
    train_data=train_df,
    time_limit=120,
    presets="medium_quality"
)

predictor


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Mon Feb  2 12:27:57 UTC 2026
CPU Count:          2
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
Memory Avail:       11.06 GB / 12.67 GB (87.3%)
Disk Space Avail:   74.76 GB / 107.72 GB (69.4%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'
Beginning AutoGluon training ... Time limit = 120s
AutoGluon will save models to "/content/AutogluonModels-bank"
Train Data Rows:    36168
Train Data Columns: 16
Label Column:       y
AutoGluon infers your prediction problem is: 'binary' (because only two unique label-values observed).
	2 unique label values:  ['no', 'yes']
	If 'binary' is not the correct problem_type, please manually specify the problem_type parameter during Predictor init (You may specify prob

###https://auto.gluon.ai/dev/tutorials/multimodal/advanced_topics/presets.html

### Leaderboard y mejor modelo

In [7]:
leaderboard = predictor.leaderboard(test_df, silent=True)
leaderboard.head(10)

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,0.935317,0.938600,roc_auc,1.205997,0.361091,104.823120,0.005225,0.000880,0.076913,2,True,6
1,LightGBMXT,0.934500,0.937725,roc_auc,0.379256,0.127201,5.757573,0.379256,0.127201,5.757573,1,True,1
2,CatBoost,0.933659,0.934310,roc_auc,0.107381,0.015882,83.171392,0.107381,0.015882,83.171392,1,True,5
3,LightGBM,0.932201,0.936678,roc_auc,0.189699,0.051869,2.360533,0.189699,0.051869,2.360533,1,True,2
4,RandomForestEntr,0.925856,0.924303,roc_auc,0.640816,0.154479,13.844213,0.640816,0.154479,13.844213,1,True,4
5,RandomForestGini,0.924398,0.924951,roc_auc,0.524435,0.165259,13.456709,0.524435,0.165259,13.456709,1,True,3


In [8]:
predictor._trainer.get_model_names()

['LightGBMXT',
 'LightGBM',
 'RandomForestGini',
 'RandomForestEntr',
 'CatBoost',
 'WeightedEnsemble_L2']

In [9]:
predictor.model_best

'WeightedEnsemble_L2'

In [10]:
best_model = predictor.model_best
G = predictor._trainer.model_graph

parents = list(G.predecessors(best_model))
print("Mejor modelo:", best_model)
print("Modelos base:", parents)


Mejor modelo: WeightedEnsemble_L2
Modelos base: ['LightGBMXT', 'LightGBM', 'RandomForestGini', 'CatBoost']


In [11]:
X_transformed = predictor.transform_features(test_df)
X_transformed.head()


,age,default,balance,housing,loan,day,duration,campaign,pdays,previous,job,marital,education,contact,month,poutcome
35806,41,0,-170,1,0,8,125,1,-1,0,7,0,1,0,8,3
1406,46,0,846,1,0,8,369,2,-1,0,0,0,2,2,8,3
11683,56,0,1860,1,0,20,129,5,-1,0,7,1,1,2,6,3
18796,37,0,2066,1,1,31,34,5,-1,0,1,1,1,0,5,3
30532,31,0,1738,0,0,5,895,3,-1,0,1,1,0,1,3,3


### Evaluación

In [12]:
perf = predictor.evaluate(test_df, silent=True)
perf

{'roc_auc': np.float64(0.9353171648637035),
 'accuracy': 0.9100962070109477,
 'balanced_accuracy': np.float64(0.7322166562304321),
 'mcc': np.float64(0.5218653552278686),
 'f1': 0.5654730090860502,
 'precision': 0.6506765067650676,
 'recall': 0.5}

### Predicciones

In [13]:
y_pred = predictor.predict(test_df)
y_pred.head()

,y
35806,no
1406,no
11683,no
18796,no
30532,yes


In [14]:
proba = predictor.predict_proba(test_df)
proba.head()

,no,yes
35806,0.998556,0.001444
1406,0.983957,0.016043
11683,0.998478,0.001522
18796,0.999746,0.000254
30532,0.475927,0.524073


### Importancia de variables

In [15]:
fi = predictor.feature_importance(test_df)
fi.head(20)

Computing feature importance via permutation shuffling for 16 features using 5000 rows with 5 shuffle sets...
	62.98s	= Expected runtime (12.6s per shuffle set)
	45.24s	= Actual runtime (Completed 5 of 5 shuffle sets)


,importance,stddev,p_value,n,p99_high,p99_low
duration,0.210423,0.009736,5.484859e-07,5,0.230470,0.190375
month,0.080650,0.005932,3.487521e-06,5,0.092865,0.068435
contact,0.035335,0.003073,6.795743e-06,5,0.041662,0.029007
day,0.020009,0.001458,3.356786e-06,5,0.023010,0.017007
poutcome,0.014727,0.001313,7.493063e-06,5,0.017430,0.012024
housing,0.009206,0.001732,1.433996e-04,5,0.012772,0.005640
pdays,0.003579,0.000587,8.397019e-05,5,0.004788,0.002369
age,0.002828,0.000668,3.479541e-04,5,0.004205,0.001452
balance,0.002738,0.000918,1.312831e-03,5,0.004628,0.000848
marital,0.001478,0.000792,6.995968e-03,5,0.003109,-0.000153


### Guardar y cargar el predictor

In [16]:
loaded = TabularPredictor.load("AutogluonModels-bank")
loaded

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
